# Fake Jobs – Exp 1: AnoLLM
- LoRA-Finetuning Qwen2.5-0.5B auf serialisierten Zeilen, Score = NLL
- Exp 1 = unsupervised: Training auf vollem Train ohne Labels; lesbare Rohwerte + Freitexte

In [ ]:
import sys, time
import numpy as np
import pandas as pd
import mlflow
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve, auc as sk_auc
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())
sys.path.insert(0, "../../anollm_src")
from anollm.anollm import AnoLLM

## Daten laden
- Rohdaten einlesen (Aufbereitung folgt in der nächsten Zelle)

In [2]:
df = pd.read_csv("../../data/raw/fake_job_postings.csv")
print("geladen:", df.shape)

geladen: (17880, 18)


## Spalten droppen & aufbereiten
- ID/Label droppen, NaN behandeln (Kategorien → "missing", Texte → ""), Split

In [3]:
text_cols = ["title", "company_profile", "description", "requirements", "benefits"]
cat_cols = ["location", "department", "salary_range", "employment_type",
            "required_experience", "required_education", "industry", "function"]

y = df["fraudulent"].values
df = df.drop(columns=["job_id", "fraudulent"])  # ID + Label droppen (übrige Spalten sind lesbar & sinnvoll)
df[cat_cols] = df[cat_cols].fillna("missing")
df[text_cols] = df[text_cols].fillna("")
feat = df  # Kategorien als Strings, Binär 0/1, location/salary_range roh + 5 Freitexte

idx = np.arange(len(feat))
tr, te = train_test_split(idx, test_size=0.3, stratify=y, random_state=42)
df_train = feat.iloc[tr].reset_index(drop=True)
df_test = feat.iloc[te].reset_index(drop=True)
y_test = y[te]
print("train", df_train.shape, "test", df_test.shape, "test outlier rate", round(y_test.mean(), 4))

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("fake_jobs_experiment_1")

train (12516, 16) test (5364, 16) test outlier rate 0.0485


/home/debian/TFM_master_thesis/.venv/lib/python3.14/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///home/debian/TFM_master_thesis/fake_job_notebooks/exp1/../../mlruns/135783001714284893', creation_time=1780130098860, experiment_id='135783001714284893', last_update_time=1780130098860, lifecycle_stage='active', name='fake_jobs_experiment_1', tags={}, trace_location=None, workspace='default'>

## AnoLLM trainieren & Scores (NLL)
- `max_length_dict` begrenzt Textspalten (Token-Budget); Vorzeichen-Auto-Korrektur

## Single-GPU-Setup (kein DDP/NCCL)
- Verteilte Env-Variablen entfernen → HF Trainer wrappt nicht in DistributedDataParallel

In [4]:
import os
import torch.distributed as dist
import anollm.anollm_trainer
from torch.utils.data import DataLoader, SequentialSampler

# 1. Alle Cluster-Variablen restlos löschen
for key in ["LOCAL_RANK", "RANK", "WORLD_SIZE", "MASTER_ADDR", "MASTER_PORT"]:
    os.environ.pop(key, None)

# 2. Falls noch eine alte Prozessgruppe aktiv ist, sauber beenden
if dist.is_available() and dist.is_initialized():
    dist.destroy_process_group()

# 3. MONKEY-PATCH: Wir definieren einen sauberen Single-GPU Dataloader
def single_gpu_get_train_dataloader(self):
    return DataLoader(
        self.train_dataset,
        batch_size=self._train_batch_size,
        sampler=SequentialSampler(self.train_dataset),
        collate_fn=self.data_collator,
        drop_last=True
    )

# 4. Die fehlerhafte Funktion der Amazon-Bibliothek im RAM überschreiben
anollm.anollm_trainer.AnoLLMTrainer.get_train_dataloader = single_gpu_get_train_dataloader

print("🚀 Single-GPU-Modus aktiv & Trainer erfolgreich für Single-GPU gepatcht!")

🚀 Single-GPU-Modus aktiv & Trainer erfolgreich für Single-GPU gepatcht!


In [ ]:
max_len = {c: 64 for c in text_cols}
model = AnoLLM(llm="Qwen/Qwen2.5-0.5B", efficient_finetuning="lora", textual_columns=text_cols,
               max_length_dict=max_len, batch_size=2, max_steps=2000, learning_rate=5e-4)

t0 = time.perf_counter()
model.fit(df_train)
scores = model.decision_function(df_test, n_permutations=8, batch_size=2, device="cuda").mean(axis=1)
runtime = time.perf_counter() - t0

scores = np.asarray(scores).astype(float)
auc = roc_auc_score(y_test, scores)
if auc < 0.5:
    scores = -scores
    auc = roc_auc_score(y_test, scores)
ap = average_precision_score(y_test, scores)
prec, rec, _ = precision_recall_curve(y_test, scores)
aucpr = sk_auc(rec, prec)

with mlflow.start_run(run_name="anollm"):
    mlflow.log_params({"llm": "Qwen/Qwen2.5-0.5B", "max_steps": 2000, "n_permutations": 8})
    mlflow.log_metric("average_precision", ap)
    mlflow.log_metric("auc_roc", auc)
    mlflow.log_metric("aucpr", aucpr)
    mlflow.log_metric("runtime_s", runtime)
print(f"anollm: AP={ap:.4f} AUCPR={aucpr:.4f} AUC={auc:.4f} time={runtime:.1f}s")

In [5]:
print(scores)

[22.37698746 45.34597301 41.85630322 ... 45.46012592 28.55211353
 36.86385965]


In [6]:
print(auc)

0.5857992223293947


In [7]:
print(ap)

0.09075442730137813
